In [1]:
"""
NCAA March Madness 三层多模型融合竞赛方案
基于notebook特征工程：Elo、进阶统计、Massey排名、历史对战等
第一层：3个FFM + 1个CatBoost + 3个seed不同的LR + XGBoost + NN
"""
import numpy as np
import pandas as pd
import os
import glob
from scipy.stats import rankdata, gmean
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = r'/kaggle/input/march-machine-learning-mania-2026'

# ==================== 1. Elo评分计算 ====================
def compute_elo(games_df, k=20, initial=1500, carry_pct=0.75):
    """跨赛季衰减 + 胜利 margin multiplier"""
    elo, records, prev_season = {}, [], None
    
    for row in games_df.sort_values(['Season', 'DayNum']).itertuples():
        season = int(row.Season)
        if prev_season is not None and season != prev_season:
            for key in elo:
                elo[key] = initial + carry_pct * (elo[key] - initial)
        prev_season = season
        
        w, l = int(row.WTeamID), int(row.LTeamID)
        elo.setdefault((season, w), initial)
        elo.setdefault((season, l), initial)
        
        ew = 1 / (1 + 10 ** ((elo[(season, l)] - elo[(season, w)]) / 400))
        mov = np.log(abs(row.WScore - row.LScore) + 1) * (2.2 / (ew * 0.001 + 1.0))
        d = k * mov * (1 - ew)
        
        elo[(season, w)] += d
        elo[(season, l)] -= d
        
        records += [(season, w, row.DayNum, elo[(season, w)]),
                    (season, l, row.DayNum, elo[(season, l)])]
    
    df = pd.DataFrame(records, columns=['Season', 'TeamID', 'DayNum', 'Elo'])
    return (df.sort_values('DayNum')
              .groupby(['Season', 'TeamID'])['Elo'].last()
              .reset_index().rename(columns={'Elo': 'FinalElo'}))

# ==================== 2. 进阶球队统计 ====================
def compute_team_stats(games_df, last_n=14):
    """全场统计 + 最近N场状态 + 对手调整"""
    def side(df, p, op):
        return pd.DataFrame({
            'Season': df['Season'].values, 'DayNum': df['DayNum'].values,
            'TeamID': df[f'{p}TeamID'].values,
            'Score': df[f'{p}Score'].values, 'OppScore': df[f'{op}Score'].values,
            'FGM': df[f'{p}FGM'].values, 'FGA': df[f'{p}FGA'].values,
            'FGM3': df[f'{p}FGM3'].values, 'FGA3': df[f'{p}FGA3'].values,
            'FTM': df[f'{p}FTM'].values, 'FTA': df[f'{p}FTA'].values,
            'OR': df[f'{p}OR'].values, 'DR': df[f'{p}DR'].values,
            'Ast': df[f'{p}Ast'].values, 'TO': df[f'{p}TO'].values,
            'Stl': df[f'{p}Stl'].values, 'Blk': df[f'{p}Blk'].values,
            'PF': df[f'{p}PF'].values,
            'OppFGM': df[f'{op}FGM'].values, 'OppFGA': df[f'{op}FGA'].values,
            'OppFGM3': df[f'{op}FGM3'].values, 'OppOR': df[f'{op}OR'].values,
            'OppTO': df[f'{op}TO'].values,
            'Win': float(p == 'W'),
        })
    
    df = pd.concat([side(games_df,'W','L'), side(games_df,'L','W')], ignore_index=True)
    e = 1e-6
    
    df['Poss'] = df['FGA'] - df['OR'] + df['TO'] + 0.44*df['FTA']
    df['OPoss'] = df['OppFGA'] - df['OppOR'] + df['OppTO'] + e
    df['eFG'] = (df['FGM'] + 0.5*df['FGM3']) / (df['FGA']+e)
    df['TS'] = df['Score'] / (2*(df['FGA'] + 0.44*df['FTA'])+e)
    df['3PAR'] = df['FGA3'] / (df['FGA']+e)
    df['FTR'] = df['FTA'] / (df['FGA']+e)
    df['AstR'] = df['Ast'] / (df['Poss']+e)
    df['TOVR'] = df['TO'] / (df['Poss']+e)
    df['ORR'] = df['OR'] / (df['OR'] + df['OppFGA'] - df['OppFGM']+e)
    df['DRR'] = df['DR'] / (df['DR'] + df['OppOR']+e)
    df['OffRtg'] = df['Score'] / (df['Poss']+e) * 100
    df['DefRtg'] = df['OppScore'] / (df['Poss']+e) * 100
    df['NetRtg'] = df['OffRtg'] - df['DefRtg']
    df['Pyth'] = df['Score']**11.5 / (df['Score']**11.5 + df['OppScore']**11.5+e)
    df['Margin'] = df['Score'] - df['OppScore']
    df['FG2M'] = df['FGM'] - df['FGM3']
    df['FG2A'] = df['FGA'] - df['FGA3']
    df['FG2Pct'] = df['FG2M'] / (df['FG2A']+e)
    df['FG3Pct'] = df['FGM3'] / (df['FGA3']+e)
    df['FTPct'] = df['FTM'] / (df['FTA']+e)
    df['BlkR'] = df['Blk'] / (df['OppFGA'] - df['OppFGM3']+e)
    df['StlR'] = df['Stl'] / (df['OPoss']+e)
    
    df = df.sort_values(['Season','TeamID','DayNum'])
    
    STAT = ['Win','Score','OppScore','eFG','TS','3PAR','FTR','AstR','TOVR',
            'ORR','DRR','OffRtg','DefRtg','NetRtg','Pyth','Margin',
            'FG2Pct','FG3Pct','FTPct','BlkR','StlR']
    
    full = df.groupby(['Season','TeamID'])[STAT].agg(['mean','std']).reset_index()
    full.columns = ['Season','TeamID'] + [f'{c}_{a}' for c,a in full.columns[2:]]
    
    rec = (df.groupby(['Season','TeamID']).tail(last_n)
             .groupby(['Season','TeamID'])
             [['Win','NetRtg','Margin','eFG','OffRtg','DefRtg','TOVR','ORR']]
             .mean().reset_index()
             .rename(columns={c: f'Rec_{c}' for c in
                               ['Win','NetRtg','Margin','eFG','OffRtg','DefRtg','TOVR','ORR']}))
    
    opp_def = df.groupby(['Season','TeamID'])['DefRtg'].mean().reset_index()
    opp_def.columns = ['Season','OppID','OppDefRtg']
    
    opp_map = pd.concat([
        games_df[['Season','WTeamID','LTeamID']].rename(columns={'WTeamID':'TeamID','LTeamID':'OppID'}),
        games_df[['Season','WTeamID','LTeamID']].rename(columns={'LTeamID':'TeamID','WTeamID':'OppID'}),
    ])
    opp_map = opp_map.merge(opp_def, on=['Season','OppID'], how='left')
    sos = opp_map.groupby(['Season','TeamID'])['OppDefRtg'].mean().reset_index()
    sos.columns = ['Season','TeamID','SOS_OppDefRtg']
    
    stats = full.merge(rec, on=['Season','TeamID'], how='left')
    stats = stats.merge(sos, on=['Season','TeamID'], how='left')
    return stats

# ==================== 3. Massey排名 ====================
def compute_massey(massey_df):
    """Massey排名系统的均值、最小值、标准差"""
    late = massey_df[massey_df['RankingDayNum'] >= 128]
    return (late.groupby(['Season','TeamID'])['OrdinalRank']
                .agg(MasseyMean='mean', MasseyMin='min',
                     MasseyMedian='median', MasseyStd='std')
                .reset_index())

# ==================== 4. 教练记录 ====================
def compute_coach_records(coaches_df, tourney_df):
    """计算教练的历史锦标赛胜率"""
    active = coaches_df[coaches_df['LastDayNum'] >= 132].copy()
    
    coach_season = active[['Season','TeamID','CoachName']].drop_duplicates(
        subset=['Season','TeamID'])
    
    tw = tourney_df.merge(
        coach_season.rename(columns={'TeamID':'WTeamID','CoachName':'WCoach'}),
        on=['Season','WTeamID'], how='left')
    tl = tourney_df.merge(
        coach_season.rename(columns={'TeamID':'LTeamID','CoachName':'LCoach'}),
        on=['Season','LTeamID'], how='left')
    
    coach_wins = tw.groupby('WCoach').size().reset_index(name='TWins')
    coach_losses = tl.groupby('LCoach').size().reset_index(name='TLosses')
    coach_wins.rename(columns={'WCoach':'Coach'}, inplace=True)
    coach_losses.rename(columns={'LCoach':'Coach'}, inplace=True)
    
    crec = coach_wins.merge(coach_losses, on='Coach', how='outer').fillna(0)
    crec['CoachTWR'] = crec['TWins'] / (crec['TWins'] + crec['TLosses'] + 1e-6)
    
    coach_twr = crec.set_index('Coach')['CoachTWR'].to_dict()
    coach_map = coach_season.set_index(['Season','TeamID'])['CoachName'].to_dict()
    
    result = {}
    for (s, tid), cname in coach_map.items():
        result[(s, tid)] = coach_twr.get(cname, 0.5)
    
    return result

# ==================== 5. 历史对战记录 ====================
def compute_h2h(tourney_games):
    """锦标赛历史对战：交锋次数和胜率"""
    df = tourney_games.copy()
    df['T1'] = df[['WTeamID','LTeamID']].min(axis=1)
    df['T2'] = df[['WTeamID','LTeamID']].max(axis=1)
    df['T1Win'] = (df['WTeamID'] == df['T1']).astype(float)
    return (df.groupby(['T1','T2'])['T1Win']
              .agg(H2H_Games='count', H2H_WinRate='mean')
              .reset_index())

# ==================== 5. 联盟特征 ====================
def compute_conf_features(conf_df):
    """联盟信息"""
    return conf_df[['Season','TeamID','ConfAbbrev']].copy()

# ==================== 6. 构建比赛特征 ====================
def build_features(matchups, seeds, elo_df, stats_df, massey_df, h2h_df, conf_df, coach_df=None, is_train=True):
    """构建所有比赛特征"""
    df = matchups.copy()
    
    if is_train:
        df['T1'] = df[['WTeamID','LTeamID']].min(axis=1)
        df['T2'] = df[['WTeamID','LTeamID']].max(axis=1)
        df['Label'] = (df['WTeamID'] == df['T1']).astype(float)
    else:
        df['T1'] = df['Team1'].astype(int)
        df['T2'] = df['Team2'].astype(int)
    
    df['Season'] = df['Season'].astype(int)
    
    df['Seed1'] = df.apply(lambda r: seeds.get(f"{r['Season']}_{r['T1']}", 16), axis=1)
    df['Seed2'] = df.apply(lambda r: seeds.get(f"{r['Season']}_{r['T2']}", 16), axis=1)
    df['SeedDiff'] = df['Seed1'] - df['Seed2']
    df['SeedSum'] = df['Seed1'] + df['Seed2']
    df['SeedWinProb'] = 1 / (1 + np.exp(0.4 * df['SeedDiff']))
    df['UpsetPot'] = (df['Seed1'] > df['Seed2']).astype(float)
    
    df = df.merge(elo_df.rename(columns={'TeamID':'T1','FinalElo':'Elo1'}),
                  on=['Season','T1'], how='left')
    df = df.merge(elo_df.rename(columns={'TeamID':'T2','FinalElo':'Elo2'}),
                  on=['Season','T2'], how='left')
    df['EloDiff'] = df['Elo1'] - df['Elo2']
    df['EloWinProb'] = 1 / (1 + 10**(-df['EloDiff']/400))
    
    sc = [c for c in stats_df.columns if c not in ['Season','TeamID']]
    df = df.merge(stats_df.rename(columns={'TeamID':'T1',**{c:f'T1_{c}' for c in sc}}),
                  on=['Season','T1'], how='left')
    df = df.merge(stats_df.rename(columns={'TeamID':'T2',**{c:f'T2_{c}' for c in sc}}),
                  on=['Season','T2'], how='left')
    for c in sc:
        df[f'D_{c}'] = df[f'T1_{c}'] - df[f'T2_{c}']
    
    if massey_df is not None:
        mc = [c for c in massey_df.columns if c not in ['Season','TeamID']]
        df = df.merge(massey_df.rename(columns={'TeamID':'T1',**{c:f'M1_{c}' for c in mc}}),
                      on=['Season','T1'], how='left')
        df = df.merge(massey_df.rename(columns={'TeamID':'T2',**{c:f'M2_{c}' for c in mc}}),
                      on=['Season','T2'], how='left')
        for c in mc:
            df[f'MD_{c}'] = df[f'M1_{c}'] - df[f'M2_{c}']
    
    df = df.merge(h2h_df, on=['T1','T2'], how='left')
    df['H2H_Games'] = df['H2H_Games'].fillna(0)
    df['H2H_WinRate'] = df['H2H_WinRate'].fillna(0.5)
    
    if coach_df is not None:
        df['CoachTWR1'] = df.apply(lambda r: coach_df.get((int(r['Season']), int(r['T1'])), 0.5), axis=1)
        df['CoachTWR2'] = df.apply(lambda r: coach_df.get((int(r['Season']), int(r['T2'])), 0.5), axis=1)
        df['D_CoachTWR'] = df['CoachTWR1'] - df['CoachTWR2']
    
    if conf_df is not None:
        df = df.merge(conf_df.rename(columns={'TeamID':'T1','ConfAbbrev':'Conf1'}),
                      on=['Season','T1'], how='left')
        df = df.merge(conf_df.rename(columns={'TeamID':'T2','ConfAbbrev':'Conf2'}),
                      on=['Season','T2'], how='left')
        df['SameConf'] = (df['Conf1'] == df['Conf2']).astype(float)
    
    df['Elo_x_SeedDiff'] = df['EloDiff'] * df['SeedDiff']
    df['NetRtg_x_SeedDiff'] = df['D_NetRtg_mean'] * df['SeedDiff']
    df['Elo_x_NetRtg'] = df['EloDiff'] * df['D_NetRtg_mean']
    df['Massey_x_Elo'] = (df.get('MD_MasseyMean', 0) * df['EloDiff'])
    
    df['UpsetScore'] = np.abs(df['SeedDiff']) * (1 - np.abs(df['EloDiff']) / 200)
    df['HotnessScore'] = np.abs(df['SeedDiff']) * df.get('D_Win_mean', pd.Series(0.5, index=df.index))
    df['ExperienceGap'] = df.get('D_Rec_Win_mean', pd.Series(0, index=df.index)) * df['SeedDiff']
    
    return df

# ==================== 7. 数据加载 ====================
def load_competition_data(data_path):
    """加载Kaggle比赛数据并构建特征"""
    print("加载数据文件...")
    
    files = glob.glob(os.path.join(data_path, '**'), recursive=True)
    data = {os.path.basename(p).split('.')[0]: pd.read_csv(p, encoding='latin-1')
             for p in files if p.endswith('.csv')}
    
    seeds_df = pd.concat([data['MNCAATourneySeeds'], data['WNCAATourneySeeds']])
    seeds = {f"{int(r.Season)}_{int(r.TeamID)}": int(r.Seed[1:3])
             for r in seeds_df.itertuples()}
    
    rs = pd.concat([data['MRegularSeasonDetailedResults'],
                    data['WRegularSeasonDetailedResults']], ignore_index=True)
    tg = pd.concat([data['MNCAATourneyDetailedResults'],
                    data['WNCAATourneyDetailedResults']], ignore_index=True)
    
    print("计算Elo评分...")
    elo_df = compute_elo(rs)
    
    print("计算球队进阶统计...")
    stats_df = compute_team_stats(rs)
    
    massey_df = compute_massey(data['MMasseyOrdinals']) if 'MMasseyOrdinals' in data else None
    
    print("计算历史对战...")
    h2h_df = compute_h2h(tg)
    
    print("计算教练记录...")
    coach_df = None
    if 'MTeamCoaches' in data:
        coach_df = compute_coach_records(data['MTeamCoaches'], tg)
    
    conf_df = None
    if 'MTeamConferences' in data and 'WTeamConferences' in data:
        conf_df = pd.concat([data['MTeamConferences'], data['WTeamConferences']])
    
    print("构建训练特征...")
    train_df = build_features(tg, seeds, elo_df, stats_df, massey_df, h2h_df, conf_df, coach_df, is_train=True)
    
    if 'SampleSubmissionStage1' in data:
        sub = data['SampleSubmissionStage1'].copy()
        sub['Season'] = sub['ID'].str.split('_').str[0].astype(int)
        sub['Team1'] = sub['ID'].str.split('_').str[1].astype(int)
        sub['Team2'] = sub['ID'].str.split('_').str[2].astype(int)
        print("构建提交特征...")
        sub_df = build_features(sub, seeds, elo_df, stats_df, massey_df, h2h_df, conf_df, coach_df, is_train=False)
        sub_ids = sub['ID'].values
    else:
        sub_df = None
        sub_ids = None
    
    EXCLUDE = {'Season','T1','T2','WTeamID','LTeamID','WScore','LScore',
               'DayNum','NumOT','WLoc','ST','Label','Team1','Team2','ID',
               'Pred','Conf1','Conf2'}
    
    # 只使用训练集和提交集共有的特征列（确保训练和预测一致）
    if sub_df is not None:
        train_cols = set(train_df.columns)
        sub_cols = set(sub_df.columns)
        common_cols = list(train_cols & sub_cols)
        
        feature_cols = [c for c in common_cols 
                      if c not in EXCLUDE 
                      and pd.api.types.is_numeric_dtype(train_df[c])]
        print(f"共同特征数: {len(feature_cols)}")
    else:
        feature_cols = [c for c in train_df.columns
                        if c not in EXCLUDE
                        and pd.api.types.is_numeric_dtype(train_df[c])]
    
    print(f"总特征数: {len(feature_cols)}")
    
    return train_df, sub_df, feature_cols, sub_ids, seeds

# ==================== 8. 简化的FFM实现 ====================
class FastFFM:
    def __init__(self, n_fields, n_features, k=8, learning_rate=0.01, 
                 reg_lambda=0.001, n_epochs=30, batch_size=512):
        self.n_fields = n_fields
        self.n_features = n_features
        self.k = k
        self.lr = learning_rate
        self.reg = reg_lambda
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        
        self.w0 = 0.0
        self.w = np.zeros(n_features)
        self.v = np.random.normal(0, 0.001, (n_features, n_fields, k))
    
    def prepare_ffm_data(self, X):
        n_samples, n_features = X.shape
        X_indices = []
        X_fields = []
        X_values = []
        
        for i in range(n_samples):
            indices = []
            fields = []
            values = []
            
            for j in range(n_features):
                if X[i, j] != 0:
                    indices.append(j)
                    fields.append(j % self.n_fields)
                    values.append(float(X[i, j]))
            
            if len(indices) > 100:
                selected = np.random.choice(len(indices), 100, replace=False)
                indices = [indices[idx] for idx in selected]
                fields = [fields[idx] for idx in selected]
                values = [values[idx] for idx in selected]
            
            X_indices.append(indices)
            X_fields.append(fields)
            X_values.append(values)
        
        return X_indices, X_fields, X_values
    
    def predict_batch(self, X_indices, X_fields, X_values):
        n_samples = len(X_indices)
        predictions = np.zeros(n_samples)
        
        for i in range(n_samples):
            indices = X_indices[i]
            fields = X_fields[i]
            values = X_values[i]
            
            score = self.w0
            for idx, val in zip(indices, values):
                if idx < len(self.w):
                    score += self.w[idx] * val
            
            n_feat = len(indices)
            if n_feat > 1:
                interactions = 0
                max_interactions = min(50, n_feat * (n_feat - 1) // 2)
                count = 0
                
                for j in range(n_feat):
                    for l in range(j + 1, n_feat):
                        if count >= max_interactions:
                            break
                        idx_j, idx_l = indices[j], indices[l]
                        field_j, field_l = fields[j], fields[l]
                        val_j, val_l = values[j], values[l]
                        
                        if (idx_j < self.v.shape[0] and idx_l < self.v.shape[0] and
                            field_j < self.n_fields and field_l < self.n_fields):
                            vj = self.v[idx_j, field_l, :]
                            vl = self.v[idx_l, field_j, :]
                            interactions += np.dot(vj, vl) * val_j * val_l
                            count += 1
                
                score += interactions
            
            predictions[i] = 1 / (1 + np.exp(-np.clip(score, -10, 10)))
        
        return predictions
    
    def fit(self, X_indices, X_fields, X_values, y):
        n_samples = len(X_indices)
        
        for epoch in range(self.n_epochs):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            
            epoch_loss = 0
            for start in range(0, n_samples, self.batch_size):
                end = min(start + self.batch_size, n_samples)
                batch_idx = indices[start:end]
                
                batch_y = y[batch_idx]
                
                batch_preds = []
                for i in batch_idx:
                    idx_list = X_indices[i]
                    field_list = X_fields[i]
                    val_list = X_values[i]
                    
                    score = self.w0
                    for idx, val in zip(idx_list, val_list):
                        if idx < len(self.w):
                            score += self.w[idx] * val
                    
                    interactions = 0
                    if len(idx_list) > 1:
                        n_interactions = min(20, len(idx_list))
                        for _ in range(n_interactions):
                            j, l = np.random.choice(len(idx_list), 2, replace=False)
                            idx_j, idx_l = idx_list[j], idx_list[l]
                            field_j, field_l = field_list[j], field_list[l]
                            val_j, val_l = val_list[j], val_list[l]
                            
                            if (idx_j < self.v.shape[0] and idx_l < self.v.shape[0] and
                                field_j < self.n_fields and field_l < self.n_fields):
                                vj = self.v[idx_j, field_l, :]
                                vl = self.v[idx_l, field_j, :]
                                interactions += np.dot(vj, vl) * val_j * val_l
                    
                    pred = 1 / (1 + np.exp(-score - 0.1 * interactions))
                    batch_preds.append(pred)
                
                batch_preds = np.array(batch_preds)
                
                errors = batch_preds - batch_y
                loss = np.mean(errors ** 2)
                epoch_loss += loss * len(batch_y)
                
                lr = self.lr / np.sqrt(epoch + 1)
                
                grad_w0 = np.mean(errors)
                self.w0 -= lr * grad_w0
                
                for i, idx in enumerate(batch_idx):
                    error = errors[i]
                    idx_list = X_indices[idx]
                    val_list = X_values[idx]
                    
                    for idx, val in zip(idx_list, val_list):
                        if idx < len(self.w):
                            self.w[idx] -= lr * (error * val + self.reg * self.w[idx])
            
            if (epoch + 1) % 5 == 0:
                sample_idx = np.random.choice(n_samples, min(1000, n_samples), replace=False)
                sample_preds = self.predict_batch(
                    [X_indices[i] for i in sample_idx],
                    [X_fields[i] for i in sample_idx],
                    [X_values[i] for i in sample_idx]
                )
                if len(np.unique(y[sample_idx])) > 1:
                    sample_auc = roc_auc_score(y[sample_idx], sample_preds)
                    print(f"  FFM Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss/n_samples:.6f}, AUC: {sample_auc:.4f}")

# ==================== 9. 贝叶斯优化 (Optuna TPE) ====================
def temporal_cv_score(model, X, y, seasons, min_train=80):
    """用历史赛季训练，预测当前赛季，返回logloss"""
    unique = sorted(np.unique(seasons))
    all_preds, all_true = [], []
    for ts in unique:
        tr = np.where(seasons < ts)[0]
        te = np.where(seasons == ts)[0]
        if len(tr) < min_train or len(te) == 0:
            continue
        model.fit(X[tr], y[tr])
        all_preds.append(model.predict_proba(X[te])[:, 1])
        all_true.append(y[te])
    if not all_preds:
        return 1.0
    return log_loss(np.concatenate(all_true), np.concatenate(all_preds))


def run_study(objective_fn, n_trials, label):
    """运行Optuna研究并返回最佳参数"""
    sampler = TPESampler(seed=42, n_startup_trials=10, multivariate=True)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective_fn, n_trials=n_trials, show_progress_bar=False)
    best = study.best_trial
    print(f"  ✓ {label} 最佳LL={best.value:.5f}  (共{len(study.trials)}次试验)")
    print(f"    参数: {best.params}")
    trials_df = study.trials_dataframe().sort_values('value').head(5)
    print(f"  Top-5试验:\n{trials_df[['number','value']].to_string(index=False)}\n")
    return best.params, best.value


def bayesian_opt_lgb(X, y, seasons, n_trials=60):
    print("\n── 贝叶斯优化: LightGBM ────────────────────────────────────────")
    def objective(trial):
        params = {
            'num_leaves': trial.suggest_int('num_leaves', 8, 128),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        }
        m = lgb.LGBMClassifier(n_estimators=400, random_state=42, verbose=-1, **params)
        return temporal_cv_score(m, X, y, seasons)
    return run_study(objective, n_trials, 'LightGBM')


def bayesian_opt_xgb(X, y, seasons, n_trials=50):
    print("\n── 贝叶斯优化: XGBoost ─────────────────────────────────────────")
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
        }
        m = xgb.XGBClassifier(n_estimators=300, eval_metric='logloss', random_state=42, verbosity=0, **params)
        return temporal_cv_score(m, X, y, seasons)
    return run_study(objective, n_trials, 'XGBoost')


def bayesian_opt_cat(X, y, seasons, n_trials=40):
    print("\n── 贝叶斯优化: CatBoost ────────────────────────────────────────")
    def objective(trial):
        params = {
            'depth': trial.suggest_int('depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
            'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
            'border_count': trial.suggest_int('border_count', 32, 255),
        }
        m = CatBoostClassifier(iterations=300, random_seed=42, verbose=0, **params)
        return temporal_cv_score(m, X, y, seasons)
    return run_study(objective, n_trials, 'CatBoost')


def bayesian_opt_lr(X, y, seasons, n_trials=30):
    print("\n── 贝叶斯优化: Logistic Regression ─────────────────────────────")
    def objective(trial):
        C = trial.suggest_float('C', 1e-3, 10.0, log=True)
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
        solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
        m = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=2000)
        return temporal_cv_score(m, X, y, seasons)
    return run_study(objective, n_trials, 'LogReg')


# ==================== 10. 神经网络 ====================
class NeuralNetworkWithHidden(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64], dropout_rate=0.3):
        super(NeuralNetworkWithHidden, self).__init__()
        
        self.layers = nn.ModuleList()
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            self.layers.append(nn.Linear(prev_dim, hidden_dim))
            self.layers.append(nn.BatchNorm1d(hidden_dim))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        self.output_layer = nn.Linear(prev_dim, 1)
        self.sigmoid = nn.Sigmoid()
        self.hidden_dim = hidden_dims[-1]
    
    def forward(self, x, return_hidden=False):
        hidden_features = x
        
        for i in range(0, len(self.layers), 4):
            hidden_features = self.layers[i](hidden_features)
            hidden_features = self.layers[i+1](hidden_features)
            hidden_features = self.layers[i+2](hidden_features)
            hidden_features = self.layers[i+3](hidden_features)
        
        output = self.sigmoid(self.output_layer(hidden_features)).squeeze()
        
        if return_hidden:
            return output, hidden_features
        return output

# ==================== 10. 三层融合模型 ====================
class ThreeLayerEnsemble:
    def __init__(self, n_ffm_models=3, n_fields=50, opt_params=None):
        self.n_ffm_models = n_ffm_models
        self.n_fields = n_fields
        self.opt_params = opt_params or {}
        self.ffm_models = []
        self.other_models = {}
        self.layer2_models = {}
        self.layer3_model = None
        self.layer3_weights = None
        self.fusion_type = 'linear'
    
    def augment_data(self, X, y, noise_level=0.02):
        """数据增强：添加特征噪声"""
        n_samples = X.shape[0]
        
        aug_X = [X]
        aug_y = [y]
        
        noise = np.random.normal(0, noise_level, X.shape)
        aug_X.append(X + noise)
        aug_y.append(y)
        
        flip_idx = np.random.choice(n_samples, int(n_samples * 0.3), replace=False)
        aug_X.append(-X[flip_idx])
        aug_y.append(1 - y[flip_idx])
        
        return np.vstack(aug_X), np.hstack(aug_y)
    
    def train_layer1(self, X_train, y_train, feature_names, feature_cols=None):
        """第一层：3个FFM + 1个CatBoost + 3个seed不同的LR + XGBoost + LightGBM + NN"""
        print("=" * 60)
        print("第一层：训练基模型")
        print("  - 3个FFM模型")
        print("  - 1个CatBoost模型")
        print("  - 3个LR模型（不同seed）")
        print("  - XGBoost模型")
        print("  - LightGBM模型")
        print("  - 神经网络模型")
        print("=" * 60)
        
        n_samples = X_train.shape[0]
        n_features = X_train.shape[1]
        
        seed_diff = None
        elo_diff = None
        if feature_cols is not None:
            seed_idx = None
            elo_idx = None
            for i, col in enumerate(feature_cols):
                if col == 'SeedDiff':
                    seed_idx = i
                elif col == 'EloDiff':
                    elo_idx = i
            if seed_idx is not None:
                seed_diff = X_train[:, seed_idx]
            if elo_idx is not None:
                elo_diff = X_train[:, elo_idx]
        
        # 准备FFM数据
        print("\n准备FFM数据...")
        ffm = FastFFM(n_fields=self.n_fields, n_features=n_features*2, k=6,
                      learning_rate=0.02, reg_lambda=0.001, n_epochs=15, batch_size=256)
        X_ffm_idx, X_ffm_fields, X_ffm_values = ffm.prepare_ffm_data(X_train)
        
        ffm_predictions = np.zeros((n_samples, self.n_ffm_models))
        other_predictions = {}
        
        # 1. 训练3个FFM模型
        print(f"\n训练 {self.n_ffm_models} 个FFM模型...")
        for i in range(self.n_ffm_models):
            print(f"  FFM模型 {i+1}/{self.n_ffm_models}")
            np.random.seed(42 + i)
            
            if i == 0:
                sample_idx = np.random.choice(n_samples, int(n_samples * 0.8), replace=True)
            elif i == 1:
                feat_idx = np.random.choice(n_features, int(n_features * 0.7), replace=False)
                X_subset = X_train[:, feat_idx]
                X_ffm_idx_s, X_ffm_fields_s, X_ffm_values_s = ffm.prepare_ffm_data(X_subset)
                sample_idx = np.arange(n_samples)
            else:
                sample_idx = np.random.choice(n_samples, int(n_samples * 0.9), replace=False)
            
            if i == 1:
                ffm.fit(X_ffm_idx_s, X_ffm_fields_s, X_ffm_values_s, y_train[sample_idx])
                preds = ffm.predict_batch(X_ffm_idx_s, X_ffm_fields_s, X_ffm_values_s)
            else:
                ffm.fit([X_ffm_idx[idx] for idx in sample_idx],
                        [X_ffm_fields[idx] for idx in sample_idx],
                        [X_ffm_values[idx] for idx in sample_idx],
                        y_train[sample_idx])
                preds = ffm.predict_batch(X_ffm_idx, X_ffm_fields, X_ffm_values)
            
            ffm_predictions[:, i] = preds
            self.ffm_models.append(ffm)
        
        # 2. 训练CatBoost模型
        print("\n训练CatBoost模型...")
        cat_params = self.opt_params.get('cat', {})
        if cat_params:
            cat_model = CatBoostClassifier(iterations=300, random_seed=42, verbose=0, **cat_params)
        else:
            cat_model = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)
        cat_model.fit(X_train, y_train)
        cat_preds = cat_model.predict_proba(X_train)[:, 1]
        other_predictions['cat'] = cat_preds
        self.other_models['cat'] = cat_model  # 保存模型
        
        # 3. 训练3个LR模型（不同seed）
        print("\n训练3个LR模型（不同seed）...")
        lr_params = self.opt_params.get('lr', {})
        lr_predictions = np.zeros((n_samples, 3))
        for i, seed in enumerate([42, 123, 456]):
            if lr_params:
                C = lr_params.get('C', 0.5)
                penalty = lr_params.get('penalty', 'l2')
                solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
                lr_model = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=2000, random_state=seed)
            else:
                lr_model = LogisticRegression(C=0.5, max_iter=1000, random_state=seed, solver='lbfgs')
            lr_model.fit(X_train, y_train)
            lr_preds = lr_model.predict_proba(X_train)[:, 1]
            lr_predictions[:, i] = lr_preds
            other_predictions[f'lr_{seed}'] = lr_preds
            self.other_models[f'lr_{seed}'] = lr_model  # 保存模型
        other_predictions['lr_all'] = lr_predictions
        
        # 4. XGBoost模型（带数据增强）
        print("\n训练XGBoost模型...")
        use_aug = True
        if use_aug:
            aug_X, aug_y = self.augment_data(X_train, y_train, noise_level=0.02)
            print(f"  数据增强: {len(X_train)} -> {len(aug_X)} 样本")
        else:
            aug_X, aug_y = X_train, y_train
        
        xgb_params = self.opt_params.get('xgb', {})
        if xgb_params:
            xgb_model = xgb.XGBClassifier(n_estimators=300, random_state=42, use_label_encoder=False, eval_metric='logloss', **xgb_params)
        else:
            xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, use_label_encoder=False, eval_metric='logloss')
        xgb_model.fit(aug_X, aug_y)
        xgb_preds = xgb_model.predict_proba(X_train)[:, 1]
        other_predictions['xgb'] = xgb_preds
        self.other_models['xgb'] = xgb_model
        
        # 5. LightGBM模型（添加异常处理 + 数据增强）
        print("\n训练LightGBM模型...")
        lgb_params = self.opt_params.get('lgb', {})
        try:
            if lgb_params:
                lgb_model = lgb.LGBMClassifier(n_estimators=400, random_state=42, verbose=-1, **lgb_params)
            else:
                lgb_model = lgb.LGBMClassifier(n_estimators=300, num_leaves=31, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
            if use_aug:
                lgb_model.fit(aug_X, aug_y)
            else:
                lgb_model.fit(X_train, y_train)
            lgb_preds = lgb_model.predict_proba(X_train)[:, 1]
            other_predictions['lgb'] = lgb_preds
            self.other_models['lgb'] = lgb_model
        except Exception as e:
            print(f"  LightGBM训练失败，使用XGBoost代替: {e}")
            other_predictions['lgb'] = xgb_preds
            self.other_models['lgb'] = xgb_model
        
        # 6. 神经网络模型（带Label Smoothing）
        print("\n训练神经网络...")
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        nn_model = NeuralNetworkWithHidden(
            input_dim=n_features,
            hidden_dims=[128, 64, 32],
            dropout_rate=0.3
        ).to(device)
        
        X_tensor = torch.FloatTensor(X_train).to(device)
        
        label_smoothing = 0.05
        y_smooth = y_train * (1 - label_smoothing) + 0.5 * label_smoothing
        y_tensor = torch.FloatTensor(y_smooth).to(device)
        
        criterion = nn.BCELoss()
        optimizer = optim.Adam(nn_model.parameters(), lr=0.001)
        
        nn_model.train()
        for epoch in range(30):
            optimizer.zero_grad()
            outputs = nn_model(X_tensor)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer.step()
            
            if (epoch + 1) % 10 == 0:
                with torch.no_grad():
                    preds = outputs.cpu().numpy()
                    auc = roc_auc_score(y_train, preds)
                    print(f"    Epoch {epoch+1}/30, Loss: {loss.item():.4f}, AUC: {auc:.4f}")
        
        with torch.no_grad():
            nn_preds, nn_hidden = nn_model(X_tensor, return_hidden=True)
            nn_preds = nn_preds.cpu().numpy()
            nn_hidden = nn_hidden.cpu().numpy()
        
        other_predictions['nn'] = nn_preds
        other_predictions['nn_hidden'] = nn_hidden
        
        # 更新模型字典（而不是覆盖）
        self.other_models.update({
            'cat': cat_model,
            'xgb': xgb_model,
            'nn': nn_model
        })
        
        return {
            'ffm_preds': ffm_predictions,
            'other_preds': other_predictions,
            'nn_hidden': nn_hidden,
            'seed_diff': seed_diff,
            'elo_diff': elo_diff
        }
    
    def train_layer2(self, layer1_results, y_train):
        """第二层：MoE风格双路径融合 + 专家模型"""
        print("\n" + "=" * 60)
        print("第二层：MoE风格双路径融合")
        print("=" * 60)
        
        ffm_preds = layer1_results['ffm_preds']
        other_preds = layer1_results['other_preds']
        nn_hidden = layer1_results['nn_hidden']
        seed_diff = layer1_results.get('seed_diff')
        elo_diff = layer1_results.get('elo_diff')
        
        if seed_diff is None:
            seed_diff = np.zeros(len(y_train))
        if elo_diff is None:
            elo_diff = np.zeros(len(y_train))
        
        upset_mask = (np.abs(seed_diff) > 4).astype(float) + (np.abs(elo_diff) > 50).astype(float)
        is_upset = upset_mask > 0
        is_favorite = (np.abs(seed_diff) > 6) | (np.abs(elo_diff) > 80)
        
        print(f"  样本分布: 常规={sum(~is_upset & ~is_favorite)}, 冷门={sum(is_upset)}, 强队={sum(is_favorite)}")
        
        print("路径1：隐层因子融合")
        
        path1_features = np.hstack([
            ffm_preds,
            nn_hidden,
            other_preds['lr_all'],
            other_preds['xgb'].reshape(-1, 1),
            other_preds['lgb'].reshape(-1, 1),
            other_preds['nn'].reshape(-1, 1),
            other_preds['cat'].reshape(-1, 1)
        ])
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        path1_nn = NeuralNetworkWithHidden(
            input_dim=path1_features.shape[1],
            hidden_dims=[64, 32],
            dropout_rate=0.2
        ).to(device)
        
        X_tensor = torch.FloatTensor(path1_features).to(device)
        y_tensor = torch.FloatTensor(y_train).to(device)
        
        criterion = nn.BCELoss()
        optimizer = optim.Adam(path1_nn.parameters(), lr=0.001)
        
        for epoch in range(20):
            optimizer.zero_grad()
            outputs = path1_nn(X_tensor)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer.step()
        
        path1_xgb = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.05,
            random_state=42, use_label_encoder=False, eval_metric='logloss'
        )
        path1_xgb.fit(path1_features, y_train)
        
        print("路径2：Pctr打分融合")
        
        path2_features = np.hstack([
            ffm_preds.mean(axis=1, keepdims=True),
            other_preds['lr_all'].mean(axis=1, keepdims=True),
            other_preds['xgb'].reshape(-1, 1),
            other_preds['lgb'].reshape(-1, 1),
            other_preds['nn'].reshape(-1, 1),
            other_preds['cat'].reshape(-1, 1)
        ])
        
        path2_nn = NeuralNetworkWithHidden(
            input_dim=path2_features.shape[1],
            hidden_dims=[32, 16],
            dropout_rate=0.2
        ).to(device)
        
        X_tensor2 = torch.FloatTensor(path2_features).to(device)
        
        optimizer2 = optim.Adam(path2_nn.parameters(), lr=0.001)
        for epoch in range(20):
            optimizer2.zero_grad()
            outputs = path2_nn(X_tensor2)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer2.step()
        
        path2_xgb = xgb.XGBClassifier(
            n_estimators=80, max_depth=4, learning_rate=0.05,
            random_state=42, use_label_encoder=False, eval_metric='logloss'
        )
        path2_xgb.fit(path2_features, y_train)
        
        path2_lr = LogisticRegression(C=0.5, max_iter=1000, random_state=42)
        path2_lr.fit(path2_features, y_train)
        
        print("专家模型训练：General / Upset / Favorite")
        
        path1_exp_features = np.column_stack([path1_features, seed_diff, elo_diff])
        
        expert_general = xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.05,
            random_state=42, use_label_encoder=False, eval_metric='logloss'
        )
        expert_general.fit(path1_exp_features, y_train)
        
        expert_upset = xgb.XGBClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.08,
            random_state=123, use_label_encoder=False, eval_metric='logloss'
        )
        if is_upset.sum() > 10:
            expert_upset.fit(path1_exp_features, y_train, sample_weight=upset_mask)
        else:
            expert_upset.fit(path1_exp_features, y_train)
        
        expert_favorite = xgb.XGBClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.08,
            random_state=456, use_label_encoder=False, eval_metric='logloss'
        )
        fav_mask = np.ones(len(y_train))
        fav_mask[is_favorite] = 2.0
        if is_favorite.sum() > 10:
            expert_favorite.fit(path1_exp_features, y_train, sample_weight=fav_mask)
        else:
            expert_favorite.fit(path1_exp_features, y_train)
        
        self.layer2_models = {
            'path1': {'nn': path1_nn, 'xgb': path1_xgb},
            'path2': {'nn': path2_nn, 'xgb': path2_xgb, 'lr': path2_lr},
            'experts': {'general': expert_general, 'upset': expert_upset, 'favorite': expert_favorite}
        }
        
        with torch.no_grad():
            path1_nn_pred = path1_nn(X_tensor).cpu().numpy()
            path2_nn_pred = path2_nn(X_tensor2).cpu().numpy()
        
        path1_xgb_pred = path1_xgb.predict_proba(path1_features)[:, 1]
        path2_xgb_pred = path2_xgb.predict_proba(path2_features)[:, 1]
        path2_lr_pred = path2_lr.predict_proba(path2_features)[:, 1]
        
        exp_general_pred = expert_general.predict_proba(path1_exp_features)[:, 1]
        exp_upset_pred = expert_upset.predict_proba(path1_exp_features)[:, 1]
        exp_favorite_pred = expert_favorite.predict_proba(path1_exp_features)[:, 1]
        
        layer2_preds = {
            'path1_nn': path1_nn_pred,
            'path1_xgb': path1_xgb_pred,
            'path2_nn': path2_nn_pred,
            'path2_xgb': path2_xgb_pred,
            'path2_lr': path2_lr_pred,
            'exp_general': exp_general_pred,
            'exp_upset': exp_upset_pred,
            'exp_favorite': exp_favorite_pred,
            'seed_diff': seed_diff,
            'elo_diff': elo_diff
        }
        
        return layer2_preds
    
    def rank_fusion(self, preds_dict):
        """Rank融合：把预测转成排名再平均"""
        ranks = []
        for name, preds in preds_dict.items():
            ranks.append(rankdata(preds))
        
        avg_rank = np.mean(ranks, axis=0)
        return avg_rank / avg_rank.max()
    
    def train_layer3(self, layer2_preds, y_train):
        """第三层：MoE门控融合"""
        print("\n" + "=" * 60)
        print("第三层：MoE门控融合")
        print("=" * 60)
        
        seed_diff = layer2_preds.get('seed_diff', np.zeros(len(y_train)))
        elo_diff = layer2_preds.get('elo_diff', np.zeros(len(y_train)))
        
        upset_mask = (np.abs(seed_diff) > 4).astype(float) + (np.abs(elo_diff) > 50).astype(float)
        is_upset = upset_mask > 0
        is_favorite = (np.abs(seed_diff) > 6) | (np.abs(elo_diff) > 80)
        
        layer2_preds_matrix = np.column_stack([
            layer2_preds['path1_nn'],
            layer2_preds['path1_xgb'],
            layer2_preds['path2_nn'],
            layer2_preds['path2_xgb'],
            layer2_preds['path2_lr']
        ])
        
        exp_preds_matrix = np.column_stack([
            layer2_preds['exp_general'],
            layer2_preds['exp_upset'],
            layer2_preds['exp_favorite']
        ])
        
        print("=== MoE门控加权 ===")
        
        w_general = np.ones(len(y_train))
        w_upset = np.zeros(len(y_train))
        w_favorite = np.zeros(len(y_train))
        
        for i in range(len(y_train)):
            if is_upset[i]:
                w_upset[i] = 0.6
                w_general[i] = 0.3
                w_favorite[i] = 0.1
            elif is_favorite[i]:
                w_favorite[i] = 0.6
                w_general[i] = 0.3
                w_upset[i] = 0.1
            else:
                w_general[i] = 0.6
                w_upset[i] = 0.2
                w_favorite[i] = 0.2
        
        moe_pred = (exp_preds_matrix[:, 0] * w_general + 
                    exp_preds_matrix[:, 1] * w_upset + 
                    exp_preds_matrix[:, 2] * w_favorite)
        moe_auc = roc_auc_score(y_train, moe_pred)
        print(f"MoE门控 AUC: {moe_auc:.6f}")
        
        preds_dict = {
            'path1_nn': layer2_preds['path1_nn'],
            'path1_xgb': layer2_preds['path1_xgb'],
            'path2_nn': layer2_preds['path2_nn'],
            'path2_xgb': layer2_preds['path2_xgb'],
            'path2_lr': layer2_preds['path2_lr']
        }
        rank_pred = self.rank_fusion(preds_dict)
        rank_auc = roc_auc_score(y_train, rank_pred)
        
        noise = np.random.normal(0, 0.02, layer2_preds_matrix.shape[1])
        layer2_preds_matrix_noisy = layer2_preds_matrix.copy()
        layer2_preds_matrix_noisy[:, 1] += noise[1]
        layer2_preds_matrix_noisy[:, 3] += noise[3]
        
        entropy_weight = 0.05
        
        def objective_with_entropy(w, X, y):
            pred = X.dot(w)
            pred = np.clip(pred, 1e-7, 1-1e-7)
            ll = log_loss(y, pred)
            w_normalized = w / (w.sum() + 1e-10)
            entropy = -np.sum(w_normalized * np.log(w_normalized + 1e-10))
            return ll + entropy_weight * entropy
        
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w}
        ]
        
        best_weights = None
        best_logloss = float('inf')
        
        for seed in [42, 123, 456, 789, 999]:
            np.random.seed(seed)
            w0 = np.random.dirichlet(np.ones(layer2_preds_matrix.shape[1]))
            
            result = minimize(
                objective_with_entropy, w0, 
                args=(layer2_preds_matrix_noisy, y_train),
                method='SLSQP',
                bounds=[(0.05, 1) for _ in range(layer2_preds_matrix.shape[1])],
                constraints=constraints,
                options={'maxiter': 1000, 'ftol': 1e-9}
            )
            
            weights = result.x / result.x.sum()
            pred = layer2_preds_matrix.dot(weights)
            pred = np.clip(pred, 1e-7, 1-1e-7)
            ll = log_loss(y_train, pred)
            
            if ll < best_logloss:
                best_logloss = ll
                best_weights = weights
        
        linear_pred = layer2_preds_matrix.dot(best_weights)
        linear_auc = roc_auc_score(y_train, linear_pred)
        
        combined_pred = 0.5 * linear_pred + 0.5 * moe_pred
        combined_auc = roc_auc_score(y_train, combined_pred)
        print(f"组合融合 AUC: {combined_auc:.6f}")
        
        print("=== Stacking Meta-Learner ===")
        
        stacking_features = np.column_stack([
            layer2_preds['path1_nn'],
            layer2_preds['path1_xgb'],
            layer2_preds['path2_nn'],
            layer2_preds['path2_xgb'],
            layer2_preds['path2_lr'],
            layer2_preds['exp_general'],
            layer2_preds['exp_upset'],
            layer2_preds['exp_favorite']
        ])
        
        meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
        meta_lr.fit(stacking_features, y_train)
        stacking_lr_pred = meta_lr.predict_proba(stacking_features)[:, 1]
        stacking_lr_auc = roc_auc_score(y_train, stacking_lr_pred)
        print(f"Stacking-LogReg AUC: {stacking_lr_auc:.6f}")
        
        meta_ridge = RidgeClassifier(alpha=1.0, random_state=42)
        meta_ridge.fit(stacking_features, y_train)
        stacking_ridge_pred = meta_ridge.decision_function(stacking_features)
        stacking_ridge_pred = 1 / (1 + np.exp(-stacking_ridge_pred))
        stacking_ridge_auc = roc_auc_score(y_train, stacking_ridge_pred)
        print(f"Stacking-Ridge AUC: {stacking_ridge_auc:.6f}")
        
        stacking_avg = (stacking_lr_pred + stacking_ridge_pred + moe_pred) / 3
        stacking_avg_auc = roc_auc_score(y_train, stacking_avg)
        print(f"Stacking平均 AUC: {stacking_avg_auc:.6f}")
        
        print(f"Rank融合 AUC: {rank_auc:.6f}")
        print(f"线性融合 AUC: {linear_auc:.6f}")
        
        all_aucs = {
            'rank': rank_auc, 
            'linear': linear_auc, 
            'moe': moe_auc, 
            'combined': combined_auc,
            'stacking_lr': stacking_lr_auc,
            'stacking_ridge': stacking_ridge_auc,
            'stacking_avg': stacking_avg_auc
        }
        best_method = max(all_aucs, key=all_aucs.get)
        
        if best_method == 'rank':
            print("使用: Rank融合")
            final_pred = rank_pred
            self.fusion_type = 'rank'
        elif best_method == 'linear':
            print("使用: 线性融合")
            final_pred = linear_pred
            self.fusion_type = 'linear'
            self.layer3_weights = best_weights
        elif best_method == 'moe':
            print("使用: MoE门控融合")
            final_pred = moe_pred
            self.fusion_type = 'moe'
        elif best_method == 'combined':
            print("使用: 组合融合")
            final_pred = combined_pred
            self.fusion_type = 'combined'
        elif best_method == 'stacking_lr':
            print("使用: Stacking-LogReg")
            final_pred = stacking_lr_pred
            self.fusion_type = 'stacking_lr'
            self.meta_model = meta_lr
            self.stacking_features = stacking_features
        elif best_method == 'stacking_ridge':
            print("使用: Stacking-Ridge")
            final_pred = stacking_ridge_pred
            self.fusion_type = 'stacking_ridge'
            self.meta_model = meta_ridge
            self.stacking_features = stacking_features
        else:
            print("使用: Stacking平均")
            final_pred = stacking_avg
            self.fusion_type = 'stacking_avg'
            self.meta_model = meta_lr
            self.stacking_features = stacking_features
        
        final_auc = roc_auc_score(y_train, final_pred)
        print(f"\n最终融合AUC: {final_auc:.6f}")
        
        print("\n=== Isotonic Calibration ===")
        self.iso_calibrator = IsotonicRegression(out_of_bounds='clip')
        self.iso_calibrator.fit(final_pred, y_train)
        
        cal_pred = self.iso_calibrator.predict(final_pred)
        cal_pred = np.clip(cal_pred, 0.025, 0.975)
        cal_auc = roc_auc_score(y_train, cal_pred)
        cal_logloss = log_loss(y_train, cal_pred)
        print(f"校准后 AUC: {cal_auc:.6f}, LogLoss: {cal_logloss:.6f}")
        
        self.use_calibration = cal_auc >= final_auc
        if self.use_calibration:
            print("使用Isotonic校准")
            self.final_pred_train = cal_pred
        else:
            print("不使用Isotonic校准")
            self.final_pred_train = final_pred
        
        return self.final_pred_train
    
    def train(self, X_train, y_train, feature_names, feature_cols=None):
        layer1_results = self.train_layer1(X_train, y_train, feature_names, feature_cols)
        layer2_preds = self.train_layer2(layer1_results, y_train)
        final_pred = self.train_layer3(layer2_preds, y_train)
        return final_pred
    
    def predict(self, X):
        """预测新数据"""
        n_samples = X.shape[0]
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        ffm_preds = np.zeros((n_samples, self.n_ffm_models))
        for i, ffm in enumerate(self.ffm_models):
            X_ffm_idx, X_ffm_fields, X_ffm_values = ffm.prepare_ffm_data(X)
            ffm_preds[:, i] = ffm.predict_batch(X_ffm_idx, X_ffm_fields, X_ffm_values)
        
        cat_preds = self.other_models['cat'].predict_proba(X)[:, 1]
        xgb_preds = self.other_models['xgb'].predict_proba(X)[:, 1]
        
        # 处理LightGBM可能不存在的情况
        if 'lgb' in self.other_models:
            lgb_preds = self.other_models['lgb'].predict_proba(X)[:, 1]
        else:
            lgb_preds = xgb_preds.copy()  # 用XGB代替
        
        X_tensor = torch.FloatTensor(X).to(device)
        nn_model = self.other_models['nn']
        nn_model.eval()
        with torch.no_grad():
            nn_preds, nn_hidden = nn_model(X_tensor, return_hidden=True)
            nn_preds = nn_preds.cpu().numpy()
            nn_hidden = nn_hidden.cpu().numpy()
        
        lr_preds = []
        for key in self.other_models:
            if key.startswith('lr_'):
                lr_preds.append(self.other_models[key].predict_proba(X)[:, 1])
        
        # 如果没有LR模型，使用其他模型
        if len(lr_preds) == 0:
            lr_preds = [xgb_preds]
        
        lr_preds = np.column_stack(lr_preds)
        
        path1_features = np.hstack([
            ffm_preds,
            nn_hidden,
            lr_preds,
            xgb_preds.reshape(-1, 1),
            lgb_preds.reshape(-1, 1),
            nn_preds.reshape(-1, 1),
            cat_preds.reshape(-1, 1)
        ])
        
        path2_features = np.hstack([
            ffm_preds.mean(axis=1, keepdims=True),
            lr_preds.mean(axis=1, keepdims=True),
            xgb_preds.reshape(-1, 1),
            lgb_preds.reshape(-1, 1),
            nn_preds.reshape(-1, 1),
            cat_preds.reshape(-1, 1)
        ])
        
        X_tensor1 = torch.FloatTensor(path1_features).to(device)
        X_tensor2 = torch.FloatTensor(path2_features).to(device)
        
        with torch.no_grad():
            path1_nn_pred = self.layer2_models['path1']['nn'](X_tensor1).cpu().numpy()
            path2_nn_pred = self.layer2_models['path2']['nn'](X_tensor2).cpu().numpy()
        
        path1_xgb_pred = self.layer2_models['path1']['xgb'].predict_proba(path1_features)[:, 1]
        path2_xgb_pred = self.layer2_models['path2']['xgb'].predict_proba(path2_features)[:, 1]
        path2_lr_pred = self.layer2_models['path2']['lr'].predict_proba(path2_features)[:, 1]
        
        layer2_preds_matrix = np.column_stack([
            path1_nn_pred,
            path1_xgb_pred,
            path2_nn_pred,
            path2_xgb_pred,
            path2_lr_pred
        ])
        
        layer2_preds_matrix = np.column_stack([
            path1_nn_pred,
            path1_xgb_pred,
            path2_nn_pred,
            path2_xgb_pred,
            path2_lr_pred
        ])
        
        fusion_type = getattr(self, 'fusion_type', 'linear')
        
        if fusion_type == 'rank':
            preds_dict = {
                'path1_nn': path1_nn_pred,
                'path1_xgb': path1_xgb_pred,
                'path2_nn': path2_nn_pred,
                'path2_xgb': path2_xgb_pred,
                'path2_lr': path2_lr_pred
            }
            final_pred = self.rank_fusion(preds_dict)
        elif fusion_type == 'geo':
            final_pred = gmean(layer2_preds_matrix + 1e-10, axis=1)
        elif fusion_type in ['moe', 'combined']:
            exp_preds = np.column_stack([
                self.layer2_models['experts']['general'].predict_proba(path1_features)[:, 1],
                self.layer2_models['experts']['upset'].predict_proba(path1_features)[:, 1],
                self.layer2_models['experts']['favorite'].predict_proba(path1_features)[:, 1]
            ])
            exp_avg = exp_preds.mean(axis=1)
            linear_pred = layer2_preds_matrix.dot(self.layer3_weights)
            final_pred = 0.5 * linear_pred + 0.5 * exp_avg
        elif fusion_type in ['stacking_lr', 'stacking_ridge', 'stacking_avg']:
            exp_preds = np.column_stack([
                self.layer2_models['experts']['general'].predict_proba(path1_features)[:, 1],
                self.layer2_models['experts']['upset'].predict_proba(path1_features)[:, 1],
                self.layer2_models['experts']['favorite'].predict_proba(path1_features)[:, 1]
            ])
            stacking_features = np.column_stack([
                path1_nn_pred,
                path1_xgb_pred,
                path2_nn_pred,
                path2_xgb_pred,
                path2_lr_pred,
                exp_preds[:, 0],
                exp_preds[:, 1],
                exp_preds[:, 2]
            ])
            if fusion_type == 'stacking_lr':
                final_pred = self.meta_model.predict_proba(stacking_features)[:, 1]
            elif fusion_type == 'stacking_ridge':
                final_pred = 1 / (1 + np.exp(-self.meta_model.decision_function(stacking_features)))
            else:
                ridge_pred = 1 / (1 + np.exp(-self.meta_model.decision_function(stacking_features)))
                lr_pred = self.meta_model.predict_proba(stacking_features)[:, 1]
                final_pred = (lr_pred + ridge_pred + exp_preds.mean(axis=1)) / 3
        else:
            final_pred = layer2_preds_matrix.dot(self.layer3_weights)
        
        if hasattr(self, 'use_calibration') and self.use_calibration and hasattr(self, 'iso_calibrator'):
            final_pred = self.iso_calibrator.predict(final_pred)
            final_pred = np.clip(final_pred, 0.025, 0.975)
        
        return final_pred

# ==================== 11. 按赛季的OOF验证 ====================
def temporal_oof_evaluation(ensemble_class, X, y, seasons, feature_cols=None, feature_names=None, n_ffm_models=3, opt_params=None):
    """
    严格按照赛季进行OOF验证
    每次用历史所有赛季训练，预测当前赛季 - 完全杜绝数据泄露
    """
    print("\n" + "=" * 60)
    print("按赛季的OOF验证（用历史预测未来）")
    print("=" * 60)
    
    if feature_names is None:
        feature_names = feature_cols
    
    unique_seasons = sorted(np.unique(seasons))
    print(f"所有赛季: {unique_seasons}")
    
    oof_predictions = np.full(len(y), np.nan)
    
    for i, test_season in enumerate(unique_seasons[1:], 1):
        train_seasons = unique_seasons[:i]
        
        train_mask = np.isin(seasons, train_seasons)
        test_mask = (seasons == test_season)
        
        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]
        
        if len(y_train) < 100 or len(y_test) == 0:
            continue
        
        print(f"\n赛季 {test_season}: 训练集={len(y_train)}, 测试集={len(y_test)}")
        
        ensemble = ensemble_class(n_ffm_models=n_ffm_models, n_fields=50, opt_params=opt_params)
        
        try:
            preds = ensemble.train(X_train, y_train, feature_names, feature_cols)
            
            test_pred = ensemble.predict(X_test)
            
            oof_predictions[test_mask] = test_pred
            
            test_auc = roc_auc_score(y_test, test_pred)
            test_logloss = log_loss(y_test, test_pred)
            print(f"  赛季 {test_season} - AUC: {test_auc:.4f}, LogLoss: {test_logloss:.4f}")
            
        except Exception as e:
            print(f"  赛季 {test_season} 训练失败: {e}")
            continue
    
    valid_mask = ~np.isnan(oof_predictions)
    if valid_mask.sum() > 0:
        overall_auc = roc_auc_score(y[valid_mask], oof_predictions[valid_mask])
        overall_logloss = log_loss(y[valid_mask], oof_predictions[valid_mask])
        print(f"\n=== OOF总体结果 ===")
        print(f"覆盖样本: {valid_mask.sum()}/{len(y)}")
        print(f"总体AUC: {overall_auc:.4f}")
        print(f"总体LogLoss: {overall_logloss:.4f}")
        return oof_predictions, overall_auc, overall_logloss
    
    return oof_predictions, 0.0, 999.0

# ==================== 12. 主程序 ====================
def main():
    print("NCAA March Madness 三层多模型融合方案")
    print("=" * 60)
    print(f"数据路径: {DATA_PATH}")
    print("=" * 60)
    
    train_df, sub_df, feature_cols, sub_ids, seeds = load_competition_data(DATA_PATH)
    
    X = train_df[feature_cols].values.astype(np.float32)
    y = train_df['Label'].values
    seasons = train_df['Season'].values
    
    print(f"\n数据形状: {X.shape}")
    print(f"正样本比例: {y.mean():.3f}")
    
    # 数据预处理
    print("\n数据预处理...")
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 阶段0: 贝叶斯优化 (Optuna)
    print("\n" + "=" * 60)
    print("阶段0: 贝叶斯优化 (Optuna TPE)")
    print("=" * 60)
    print("\n=== 贝叶斯优化 (Optuna TPE) ===")
    
    lgb_params, lgb_score = bayesian_opt_lgb(X, y, seasons, n_trials=60)
    xgb_params, xgb_score = bayesian_opt_xgb(X, y, seasons, n_trials=50)
    cat_params, cat_score = bayesian_opt_cat(X, y, seasons, n_trials=40)
    lr_params, lr_score = bayesian_opt_lr(X, y, seasons, n_trials=30)
    
    print(f"\n优化完成！各模型最佳LogLoss:")
    print(f"  LightGBM: {lgb_score:.5f}")
    print(f"  XGBoost:  {xgb_score:.5f}")
    print(f"  CatBoost: {cat_score:.5f}")
    print(f"  LogReg:   {lr_score:.5f}")
    
    opt_params = {
        'xgb': xgb_params,
        'cat': cat_params,
        'lr': lr_params,
        'lgb': lgb_params
    }
    
    # 按赛季的OOF验证（核心！）
    print("\n" + "=" * 60)
    print("阶段1：按赛季的OOF验证（用历史预测未来）")
    print("=" * 60)
    oof_pred, oof_auc, oof_logloss = temporal_oof_evaluation(
        ThreeLayerEnsemble, X, y, seasons, feature_cols=feature_cols, n_ffm_models=3, opt_params=opt_params
    )
    
    # 最终在全量数据上训练
    print("\n" + "=" * 60)
    print("阶段2：全量数据训练（用于预测提交）")
    print("=" * 60)
    
    ensemble = ThreeLayerEnsemble(n_ffm_models=3, n_fields=50, opt_params=opt_params)
    train_pred = ensemble.train(X, y, feature_cols, feature_cols)
    train_auc = roc_auc_score(y, train_pred)
    print(f"\n全量训练AUC: {train_auc:.6f}")
    
    # 生成提交文件
    print("\n" + "=" * 60)
    print("生成提交文件...")
    print("=" * 60)
    
    if sub_df is not None and sub_ids is not None:
        # 直接使用load_competition_data中计算的特征列
        X_sub = sub_df[feature_cols].values.astype(np.float32)
        X_sub = imputer.transform(X_sub)
        X_sub = scaler.transform(X_sub)
        
        sub_preds = ensemble.predict(X_sub)
        sub_preds = np.clip(sub_preds, 0.025, 0.975)
        
        submission = pd.DataFrame({
            'ID': sub_ids,
            'Pred': sub_preds
        })
        submission.to_csv('submission.csv', index=False)
        print(f"\n已保存 → submission.csv")
        print(f"预测统计: mean={sub_preds.mean():.3f}, std={sub_preds.std():.3f}")
        print(f"          min={sub_preds.min():.3f}, max={sub_preds.max():.3f}")
    else:
        print("警告: 缺少提交数据，无法生成submission.csv")
    
    print("\n训练完成!")
    return ensemble, imputer, scaler, feature_cols, sub_df, sub_ids, seeds

if __name__ == "__main__":
    main()

NCAA March Madness 三层多模型融合方案
数据路径: /kaggle/input/march-machine-learning-mania-2026
加载数据文件...
计算Elo评分...
计算球队进阶统计...
计算历史对战...
计算教练记录...
构建训练特征...
构建提交特征...
共同特征数: 188
总特征数: 188

数据形状: (2410, 188)
正样本比例: 0.502

数据预处理...

阶段0: 贝叶斯优化 (Optuna TPE)

=== 贝叶斯优化 (Optuna TPE) ===

── 贝叶斯优化: LightGBM ────────────────────────────────────────
  ✓ LightGBM 最佳LL=0.07947  (共60次试验)
    参数: {'num_leaves': 69, 'max_depth': 4, 'learning_rate': 0.037749937123737504, 'min_child_samples': 23, 'subsample': 0.5808960144682656, 'colsample_bytree': 0.8029372453319791, 'reg_alpha': 1.2293062326907656, 'reg_lambda': 1.5341235950609808, 'min_split_gain': 0.48309441169641765}
  Top-5试验:
 number    value
     47 0.079474
     46 0.079533
     45 0.079675
     51 0.079832
     49 0.079847


── 贝叶斯优化: XGBoost ─────────────────────────────────────────
  ✓ XGBoost 最佳LL=0.08436  (共50次试验)
    参数: {'max_depth': 6, 'learning_rate': 0.027350135065342524, 'subsample': 0.950285262431147, 'colsample_bytree': 0.768728606512447, 